# GPT follow along

we write a small language model from scratch

In [ ]:
# @title Imports
import requests
import random
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F
from math import ceil
from tqdm import tqdm

In [ ]:
# @title Hyper Parameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 200
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
p_drop = 0.0

## Data Preparation (Tiny Shakespeare)

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
# @title create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

In [ ]:
# @title data set creation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
valid_data = data[n:]
print(f'Shapes: train: {train_data.shape}, validation: {valid_data.shape}')
print(f'Train Tokens: {train_data[:100]}'[:90] + '...')
print(f'Train Text: {decode(train_data[:100].tolist())}')

In [ ]:
# @title Data Access
def get_batch(split='train'):
    data = train_data if split == 'train' else valid_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## Data Preparation (Tiny Sories)



In [ ]:
# @title Vocabulary Mapping

vocab = sorted(set(" \n!\"#$%&'()*/+,-.0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ_abcdefghijklmnopqrstuvwxyz"))
vocab_size = len(vocab)
stoi = { ch:i for i,ch in enumerate(vocab) }
itos = { i:ch for i,ch in enumerate(vocab) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [ ]:
# @title Downloads

TRAINING_ROWS = 200
VALIDATION_ROWS = 10

def get_tinystories_rows(split="train", offset=0, length=100, desc="Downloading"):
    """Fetch `length` rows from TinyStories whose text is within `vocab`, batching in chunks of 100."""
    rows = []
    fetched = 0  # raw rows consumed from the dataset (for advancing the offset)
    with tqdm(total=length, desc=desc, unit="row") as bar:
        while len(rows) < length:
            url = (
                "https://datasets-server.huggingface.co/rows"
                "?dataset=roneneldan%2FTinyStories&config=default"
                f"&split={split}&offset={offset + fetched}&length=100"
            )
            response = requests.get(url)
            response.raise_for_status()
            batch = response.json()["rows"]
            if not batch:
                break  # ran out of data in this split

            fetched += len(batch)
            for row in batch:
                text = row["row"]["text"]
                if set(text).issubset(vocab):
                    rows.append(text)
                    bar.update(1)
                    if len(rows) >= length:
                        break
    return rows[:length]

# Training data
train_text = get_tinystories_rows(split="train", offset=0, length=TRAINING_ROWS, desc="Training")

# Validation data — pick a random offset (total validation rows is approx 21k)
random.seed(42)
random_offset = random.randint(0, 21000 - VALIDATION_ROWS)
validation_text = get_tinystories_rows(split="validation", offset=random_offset, length=VALIDATION_ROWS, desc="Validation")

In [ ]:
# @title data set creation
train_data = torch.concat([torch.tensor(encode(t)) for t in train_text])
valid_data = torch.concat([torch.tensor(encode(t)) for t in validation_text])
print(f'Shapes: train: {train_data.shape}, validation: {valid_data.shape}')
print(f'Train Tokens: {train_data[:100]}'[:90] + '...')
print(f'Train Text: {decode(train_data[:100].tolist())}')

In [ ]:
# @title Data Access
def get_batch(split='train'):
    data = train_data if split == 'train' else valid_data
    ix = torch.randint(len(data) - block_size, (block_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## Training

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def dummy_train(cls, **kwargs):
    # print dummy run
    mod = cls(**kwargs).to(device)
    x, y = get_batch()
    logits, loss = mod(x, y)
    print(f'Random Loss: {loss:.4f}')
    print(f'Expected Loss: {-torch.log(torch.tensor(1/vocab_size)):.4f}')

    # generate randomly
    print('Random Text Generation:')
    starting_sequence = torch.tensor(encode('\n'), device=device).view(1, 1)
    print('>>' + decode(mod.generate(starting_sequence, max_new_tokens=50)[0].tolist()) + '<<')

    # perform quick training
    print('Training...')
    optimiser = torch.optim.AdamW(mod.parameters(), lr=learning_rate)
    with tqdm(range(max_iters)) as bar:
        for step in bar:
            if step % eval_interval == 0 or step == max_iters - 1:
                bar.set_postfix(loss=estimate_loss(mod)['train'].item())

            x, y = get_batch()

            _, loss = mod(x, y)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()

    # Estmate final loss
    print('Final Loss:')
    print(estimate_loss(mod))

    # generate randomly
    print('Random Text Generation:')
    starting_sequence = torch.tensor(encode('\n'), device=device).view(1, 1)
    print('>>' + decode(mod.generate(starting_sequence, max_new_tokens=100)[0].tolist()) + '<<')


## Model Definition

In [ ]:
# @title Bigram Model

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # B is batch, T is the seen context window, C is the vocabulary
        # idx, targets are of shape (B, T)

        # Query lookup table
        logits = self.token_embedding_table(idx) # (B, T, C)
        B, T, C = logits.shape

        # unwrap the T dimension into the batch
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is of shape (B, T)
        for _ in range(max_new_tokens):
            # predict
            logits, loss = self(idx)
            # get only the last new token
            logits = logits[:, -1, :] # (B, C)
            # normalise the logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the predicted probability space
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
#dummy_train(BigramLanguageModel)

In [ ]:
# @title Bigram v2

class BigramLanguageModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        # B is batch, T is the seen context window, C is n_embed
        # idx, targets are of shape (B, T)
        B, T = idx.shape

        # Query lookup table
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        B, T, C = logits.shape

        # unwrap the T dimension into the batch
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is of shape (B, T)
        for _ in range(max_new_tokens):
            # predict
            logits, loss = self(idx[:, -block_size:])
            # get only the last new token
            logits = logits[:, -1, :] # (B, C)
            # normalise the logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the predicted probability space
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
# @title Single Headed Self Attention

class SingleHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        # attention scores
        k = self.key(x) # (B, T, C)
        q = self.query(x) # (B, T, C)
        v = self.value(x) # (B, T, C)

        # attention scores
        aff = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # (B, T, T)
        wei = aff.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # aggregation
        wei = wei.softmax(1) # (B, T, T)
        out = wei @ v # (B, T, C)
        return out

class SingleHeadLanguage(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.sa_head = SingleHead(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        # B is batch, T is the seen context window, C is n_embed
        # idx, targets are of shape (B, T)
        B, T = idx.shape

        # get tokens from input
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)

        # single head attention
        x = self.sa_head(x) # (B, T, C)

        # project into vocab dimension
        logits = self.lm_head(x) # (B, T, vocab_size)

        # calculate loss
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is of shape (B, T)
        for _ in range(max_new_tokens):
            # predict
            logits, loss = self(idx[:, -block_size:])
            # get only the last new token
            logits = logits[:, -1, :] # (B, C)
            # normalise the logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the predicted probability space
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
# @title Multi Head Attention

class MultiHead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([SingleHead(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class MultiHeadLanguage(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.sa_head = MultiHead(4, n_embed // 4)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        # B is batch, T is the seen context window, C is n_embed
        # idx, targets are of shape (B, T)
        B, T = idx.shape

        # get tokens from input
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)

        # multi head attention
        x = self.sa_head(x) # (B, T, C)

        # project into vocab dimension
        logits = self.lm_head(x) # (B, T, vocab_size)

        # calculate loss
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is of shape (B, T)
        for _ in range(max_new_tokens):
            # predict
            logits, loss = self(idx[:, -block_size:])
            # get only the last new token
            logits = logits[:, -1, :] # (B, C)
            # normalise the logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the predicted probability space
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
# @title Feed Forward Integration

class FeedForward(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class SingleGptPass(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.sa_head = MultiHead(4, n_embed // 4)
        self.ffw = FeedForward(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        # B is batch, T is the seen context window, C is n_embed
        # idx, targets are of shape (B, T)
        B, T = idx.shape

        # get tokens from input
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)

        # multi head attention
        x = self.sa_head(x) # (B, T, C)

        # feed forward
        x = self.ffw(x)

        # project into vocab dimension
        logits = self.lm_head(x) # (B, T, vocab_size)

        # calculate loss
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is of shape (B, T)
        for _ in range(max_new_tokens):
            # predict
            logits, loss = self(idx[:, -block_size:])
            # get only the last new token
            logits = logits[:, -1, :] # (B, C)
            # normalise the logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the predicted probability space
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [ ]:
# @title Attention Is All You Need - Decoder

class AttentionHead(nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.key = nn.Linear(dim_in, dim_out, bias=False)
        self.query = nn.Linear(dim_in, dim_out, bias=False)
        self.value = nn.Linear(dim_in, dim_out, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(p_drop)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # (B, T, C)
        q = self.query(x) # (B, T, C)
        v = self.value(x) # (B, T, C)

        aff = q @ k.transpose(-2, -1) * C**-0.5 # (B, T, T)
        wei = torch.masked_fill(aff, self.tril[:T, :T] == 0, float('-inf'))
        wei = wei.softmax(-1)
        wei = self.dropout(wei)

        x = wei @ v # (B, T, C)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        head_dim = dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(dim, head_dim) for _ in range(n_heads)])
        self.proj = nn.Linear(n_heads * head_dim, dim)
        self.dropout = nn.Dropout(p_drop)

    def forward(self, x):
        out = torch.concat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForwardNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.ReLU(),
            nn.Linear(4 * dim, dim),
            nn.Dropout(p_drop)
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.norm_one = nn.LayerNorm(dim)
        self.mha = MultiHeadAttention(dim, n_heads)
        self.norm_two = nn.LayerNorm(dim)
        self.ffw = FeedForwardNet(dim)

    def forward(self, x):
        x = x + self.mha(self.norm_one(x)) # (B, T, C)
        x = x + self.ffw(self.norm_two(x)) # (B, T, C)
        return x

class Decoder(nn.Module):
    def __init__(self, num_blocks, n_heads):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.attention_blocks = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(num_blocks)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # B is batch, T is the seen context window, C is n_embed
        # idx, targets are of shape (B, T)
        B, T = idx.shape

        # get tokens from input
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)

        # multi head attention
        x = self.attention_blocks(x)
        x = self.ln_f(x)

        # project into vocab dimension
        logits = self.lm_head(x) # (B, T, vocab_size)

        # calculate loss
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is of shape (B, T)
        for _ in range(max_new_tokens):
            # predict
            logits, loss = self(idx[:, -block_size:])
            # get only the last new token
            logits = logits[:, -1, :] # (B, C)
            # normalise the logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the predicted probability space
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [ ]:
dummy_train(Decoder, num_blocks=n_layer, n_heads=n_head)

## temp

In [ ]:
# -*- coding: utf-8 -*-
"""SLM.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1xwZ8Zc8zSGVoRzqo_miKMTQBWX0Dvl7Y

## SetUp
"""

# Commented out IPython magic to ensure Python compatibility.
# @title Imports
# %pip install lightning

import re
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

"""## Model Architecure"""

# @title masked multi head attention

class MaskedMultiHeadAttention(torch.nn.Module):
  """Module for masked and unmasked self and cross attention."""
  def __init__(self, d_model, num_heads, mask, p_drop):
    super().__init__()
    self.num_heads = num_heads
    self.d_head = d_model // num_heads
    layer_dim = self.d_head * num_heads

    self.query = nn.Linear(d_model, layer_dim)
    self.key = nn.Linear(d_model, layer_dim)
    self.value = nn.Linear(d_model, layer_dim)

    if mask is not None:
      assert mask.ndim == 4, "Mask must be a 4D tensor"
      self.register_buffer('mask', mask)
    else:
      self.mask = None
    self.dropout = nn.Dropout(p_drop)
    self.proj = nn.Linear(layer_dim, d_model)

  def forward(self, qs, ks=None, vs=None, key_padding_mask=None):
    # assume single x if not provided individually
    assert (ks is None) == (vs is None), "Provide either qs alone (self-attention) or all of qs, ks, vs (cross-attention)"
    if ks is None:
      ks = qs
    if vs is None:
      vs = qs

    # the query, key and value sources are of shape batch, by time, by channels
    B, T1, C = qs.shape
    B2, T2, C2 = ks.shape
    B3, T3, C3 = vs.shape
    assert T2 == T3, "Keys and values must have the same context length"
    assert C == C2 == C3, "Keys, queries and values must have the same channels"
    assert B == B2 == B3, "Batch size must be the same"

    # get queries, keys and values
    q = self.query(qs) # B T1 C
    k = self.key(ks) # B T2 C
    v = self.value(vs) # B T2 C

    # unbatch to enable per head masked scaled dot product
    # (B, T, C) -> (B, T, H, C') -> (B, H, T, C')
    q = q.view(B, T1, self.num_heads, self.d_head).transpose(1, 2)
    k = k.view(B, T2, self.num_heads, self.d_head).transpose(1, 2)
    v = v.view(B, T2, self.num_heads, self.d_head).transpose(1, 2)

    # calculate affinities and apply masking
    affinity = q @ k.transpose(-1, -2) * self.d_head**-.5 # (B, H, T1, T2)
    if self.mask is not None:
      affinity = torch.masked_fill(affinity, self.mask[:, :, :T1, :T2] == 0, float('-inf'))
    if key_padding_mask is not None:
      # (B, T2) -> (B, 1, 1, T2); broadcast over heads and query positions
      kpm = key_padding_mask[:, None, None, :T2]
      affinity = torch.masked_fill(affinity, kpm == 0, float('-inf'))

    # calculate new value
    weights = affinity.softmax(-1)
    weights = self.dropout(weights)
    out = weights @ v # (B, H, T1, C')

    # restore shape and apply projection
    out = out.transpose(1, 2) # (B, T1, H, C')
    out = out.reshape(B, T1, -1)
    out = self.proj(out)
    return out

# @title simple feed forward network

class FeedForward(nn.Module):
  def __init__(self, d_model, p_drop):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(d_model, 4 * d_model),
      nn.ReLU(),
      nn.Linear(4 * d_model, d_model),
      nn.Dropout(p_drop),
    )
  def forward(self, x):
    return self.net(x)

# @title Encoder Block

class EncoderBlock(nn.Module):
  def __init__(self, dim, num_heads, max_context, p_drop):
    super().__init__()
    # layer norms
    self.ln_1 = nn.LayerNorm(dim)
    self.ln_2 = nn.LayerNorm(dim)

    # unmasked self attention
    self.sa_head = MaskedMultiHeadAttention(dim, num_heads, None, p_drop)
    # simple feed forwrd network
    self.ffw_net = FeedForward(dim, p_drop)

  def forward(self, x, key_padding_mask=None):
    x = x + self.sa_head(self.ln_1(x), key_padding_mask=key_padding_mask)
    x = x + self.ffw_net(self.ln_2(x))
    return x

# @title Decoder Block

class DecoderBlock(nn.Module):
  def __init__(self, dim, num_heads, max_context, p_drop):
    super().__init__()
    # layer norms
    self.ln_1 = nn.LayerNorm(dim)
    self.ln_2 = nn.LayerNorm(dim)
    self.ln_3 = nn.LayerNorm(dim)

    # masked self attention
    mask = torch.tril(torch.ones(1, 1, max_context, max_context))
    self.sa_head = MaskedMultiHeadAttention(dim, num_heads, mask, p_drop)
    # unmasked cross attention
    self.ca_head = MaskedMultiHeadAttention(dim, num_heads, None, p_drop)
    # simple feed forwrd network
    self.ffw_net = FeedForward(dim, p_drop)

  def forward(self, x, context, x_key_padding_mask=None, context_key_padding_mask=None):
    # self attention
    x = x + self.sa_head(self.ln_1(x), key_padding_mask=x_key_padding_mask)
    # cross attention
    x = x + self.ca_head(self.ln_2(x), context, context, context_key_padding_mask)
    x = x + self.ffw_net(self.ln_3(x))
    return x

# @title Full Transformer Network

class Transformer(nn.Module):
  def __init__(self, num_layers, dim, num_heads, max_context, vocab_size, p_drop):
    super().__init__()
    self.max_context = max_context
    self.tok_emb = nn.Embedding(vocab_size, dim)
    self.pos_emb = nn.Embedding(max_context, dim)

    self.encoder = nn.ModuleList([
        EncoderBlock(dim, num_heads, max_context, p_drop)
        for _ in range(num_layers)
    ])
    self.decoder = nn.ModuleList([
        DecoderBlock(dim, num_heads, max_context, p_drop)
        for _ in range(num_layers)
    ])

    self.proj = nn.Linear(dim, vocab_size)

  def forward(self, x, context, x_pad_mask=None, context_pad_mask=None):
    assert x.shape[1] <= self.max_context, "Input sequence too long"
    assert context.shape[1] <= self.max_context, "Context sequence too long"

    # generate context embeddings first
    tok = self.tok_emb(context.long())
    pos = self.pos_emb(torch.arange(context.shape[1], device=context.device))
    context = tok + pos
    for block in self.encoder:
      context = block(context, context_pad_mask)

    # generate target embeddings
    tok = self.tok_emb(x.long())
    pos = self.pos_emb(torch.arange(x.shape[1], device=x.device))
    x = tok + pos

    # perform decoder pass
    for block in self.decoder:
      x = block(x, context, x_pad_mask, context_pad_mask)

    # project embedding into vocab dimension
    logits = self.proj(x)
    return logits

  def generate(self, context, start_tokens, max_tokens):
    idx = start_tokens
    for _ in range(max_tokens):
        logits = self(idx[:, -self.max_context:], context[:, -self.max_context:])
        logits = logits[:, -1, :]
        probs = logits.softmax(dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

"""## Training data"""

# @title Vocabulary

BASE_ALPHABET = sorted(set(
    " \n!\"#$%&'()*/+,-.0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ_abcdefghijklmnopqrstuvwxyz"
))
PAD_TOK, BOS_TOK, EOS_TOK = "<pad>", "<bos>", "<eos>"
SPECIALS = [PAD_TOK, BOS_TOK, EOS_TOK]

VOCAB = SPECIALS + BASE_ALPHABET
VOCAB_SIZE = len(VOCAB)

stoi = {ch: i for i, ch in enumerate(VOCAB)}
itos = {i: ch for i, ch in enumerate(VOCAB)}

PAD_ID = stoi[PAD_TOK]
BOS_ID = stoi[BOS_TOK]
EOS_ID = stoi[EOS_TOK]

ALPHABET_SET = set(BASE_ALPHABET)  # used to reject out-of-vocab stories


def encode(s):
    # string -> list[int] over the base alphabet (no specials)
    return [stoi[c] for c in s]


def decode(ids):
    # list[int] -> string, dropping special tokens for readability
    out = []
    for i in ids:
        ch = itos[int(i)]
        if ch in SPECIALS:
            continue
        out.append(ch)
    return "".join(out)

# @title Sentence splitting + example construction

# Greedy: everything up to a run of sentence-final punctuation (with any trailing
# closing quote/bracket), or the trailing fragment if a story has no final punct.
_SENTENCE_RE = re.compile(r'[^.!?]*[.!?]+(?:["\')\]]+)?|\S[^.!?]*$')


def split_sentences(text):
    text = text.strip()
    parts = [m.group().strip() for m in _SENTENCE_RE.finditer(text)]
    return [p for p in parts if p]


def story_to_examples(text, max_context):
    """One story -> list of (context_ids, decoder_in, decoder_target).

    For every sentence i >= 1:
      context     = sentences 0 .. i-1 joined with spaces
      decoder_in  = <bos> + sentence_i
      target      =         sentence_i + <eos>   (decoder_in shifted by one)

    The TARGET sentence must fit in max_context (we can't truncate what we are
    trying to generate), so any example whose target is too long is DISCARDED.
    The CONTEXT may come from an arbitrarily long story; we keep the most recent
    max_context tokens and truncate the rest. A whole story is therefore only
    useless if every one of its sentences is over-long.
    """
    sentences = split_sentences(text)
    examples = []
    for i in range(1, len(sentences)):
        full = [BOS_ID] + encode(sentences[i]) + [EOS_ID]  # length = sentence_len + 2

        # discard: decoder_in (full[:-1]) must fit within max_context positions
        if len(full) - 1 > max_context:
            continue

        # truncate: keep the most recent max_context tokens of prior sentences
        ctx_ids = encode(" ".join(sentences[:i]))[-max_context:]
        if len(ctx_ids) == 0:
            continue

        examples.append((ctx_ids, full[:-1], full[1:]))
    return examples

# @title TinyStories streaming (100 rows/request)

ROWS_PER_SHARD = 500       # stories held in RAM at once
VALIDATION_ROWS = 100      # fixed validation shard (kept constant across swaps)
HF_FETCH_CHUNK = 100       # datasets-server hard limit per request


def get_tinystories_rows(split, offset, length, max_context):
    """Fetch `length` stories that (a) use only our alphabet and (b) yield at
    least one valid example (i.e. have at least one sentence whose target fits
    in max_context). Returns (rows, next_offset).

    Stories are filtered here too, so the shard we keep is already clean and we
    don't waste RAM on stories we'd discard entirely.
    """
    rows = []
    fetched = 0
    while len(rows) < length:
        url = (
            "https://datasets-server.huggingface.co/rows"
            "?dataset=roneneldan%2FTinyStories&config=default"
            f"&split={split}&offset={offset + fetched}&length={HF_FETCH_CHUNK}"
        )
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        batch = resp.json()["rows"]
        if not batch:
            break  # exhausted this split
        fetched += len(batch)
        for row in batch:
            text = row["row"]["text"]
            if not set(text).issubset(ALPHABET_SET):
                continue
            # keep only stories that produce at least one in-budget example
            if story_to_examples(text, max_context):
                rows.append(text)
                if len(rows) >= length:
                    break
    return rows[:length], offset + fetched

# @title Dataset + padding collate

class SentenceDataset(torch.utils.data.Dataset):
    """Flattens a list of stories into next-sentence examples."""

    def __init__(self, stories, max_context):
        self.examples = []
        for story in stories:
            self.examples.extend(story_to_examples(story, max_context))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ctx, dec_in, target = self.examples[idx]
        return (
            torch.tensor(ctx, dtype=torch.long),
            torch.tensor(dec_in, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
        )


def _pad_stack(seqs, pad_value):
    longest = max(len(s) for s in seqs)
    out = torch.full((len(seqs), longest), pad_value, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, : len(s)] = s
    return out


def collate(batch):
    # right-pad each field; build boolean padding masks (1 = real, 0 = pad)
    ctxs, dec_ins, targets = zip(*batch)
    context = _pad_stack(ctxs, PAD_ID)
    dec_in = _pad_stack(dec_ins, PAD_ID)
    target = _pad_stack(targets, PAD_ID)        # PAD positions ignored by the loss
    context_pad_mask = (context != PAD_ID).long()
    dec_in_pad_mask = (dec_in != PAD_ID).long()
    return context, dec_in, target, context_pad_mask, dec_in_pad_mask

# @title LightningDataModule with shard swapping

class TinyStoriesDataModule(pl.LightningDataModule):
    def __init__(self, rows_per_shard=ROWS_PER_SHARD, validation_rows=VALIDATION_ROWS,
                 batch_size=64, max_context=128, seed=42):
        super().__init__()
        self.rows_per_shard = rows_per_shard
        self.validation_rows = validation_rows
        self.batch_size = batch_size
        self.max_context = max_context

        # torch generator for picking the (fixed) validation offset
        self._gen = torch.Generator().manual_seed(seed)
        self._train_offset = 0        # advances as we stream through the corpus
        self.train_dataset = None
        self.val_dataset = None
        self._shard_index = 0

    def setup(self, stage=None):
        if self.val_dataset is None:
            # fixed random validation slice so val loss is comparable across swaps
            high = max(1, 21000 - self.validation_rows)
            val_offset = int(torch.randint(0, high, (1,), generator=self._gen).item())
            val_rows, _ = get_tinystories_rows(
                "validation", val_offset, self.validation_rows, self.max_context)
            self.val_dataset = SentenceDataset(val_rows, self.max_context)
            print(f"[data] validation shard: {len(val_rows)} stories "
                  f"-> {len(self.val_dataset)} examples")
        if self.train_dataset is None:
            self.load_new_train_shard()

    def load_new_train_shard(self):
        # stream a fresh 500-row shard, advancing the offset each time
        train_rows, next_offset = get_tinystories_rows(
            "train", self._train_offset, self.rows_per_shard, self.max_context)
        if not train_rows:
            # wrapped past the end -> start over from the top
            self._train_offset = 0
            train_rows, next_offset = get_tinystories_rows(
                "train", 0, self.rows_per_shard, self.max_context)
        self._train_offset = next_offset
        self._shard_index += 1
        self.train_dataset = SentenceDataset(train_rows, self.max_context)
        print(f"[data] loaded train shard #{self._shard_index}: "
              f"{len(train_rows)} stories -> {len(self.train_dataset)} examples "
              f"(next offset {self._train_offset})")

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset, batch_size=self.batch_size, shuffle=True,
            collate_fn=collate, num_workers=2, drop_last=True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset, batch_size=self.batch_size, shuffle=False,
            collate_fn=collate, num_workers=2)

"""## Training Source Code"""

# @title LightningModule (wraps the Transformer, LR decay)

class SLMLightning(pl.LightningModule):
    def __init__(self, num_layers=4, dim=128, num_heads=4, max_context=128,
                 vocab_size=VOCAB_SIZE, p_drop=0.1, learning_rate=3e-4,
                 max_steps=20000):
        super().__init__()
        self.save_hyperparameters()
        self.model = Transformer(num_layers=num_layers, dim=dim, num_heads=num_heads,
                                 max_context=max_context, vocab_size=vocab_size,
                                 p_drop=p_drop)
        self.learning_rate = learning_rate
        self.max_steps_total = max_steps

    def forward(self, dec_in, context, dec_in_pad_mask=None, context_pad_mask=None):
        return self.model(dec_in, context,
                          x_pad_mask=dec_in_pad_mask,
                          context_pad_mask=context_pad_mask)

    def _step(self, batch):
        context, dec_in, target, context_pad_mask, dec_in_pad_mask = batch
        logits = self.model(dec_in, context,
                            x_pad_mask=dec_in_pad_mask,
                            context_pad_mask=context_pad_mask)        # (B, T, V)
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target.long().reshape(-1),
            ignore_index=PAD_ID)                                      # ignore padded targets
        return loss

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        # cosine decay to ~1/10th of the base LR over the full training run
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=self.max_steps_total, eta_min=self.learning_rate * 0.1)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "interval": "step"},
        }

    @torch.no_grad()
    def generate_next_sentence(self, context_text, max_tokens=None):
        # convenience helper: given a context string, sample the next sentence
        self.eval()
        max_tokens = max_tokens or self.hparams.max_context
        ctx_ids = encode(context_text)[-self.hparams.max_context:]
        context = torch.tensor([ctx_ids], device=self.device)
        ctx_mask = torch.ones_like(context)
        out = torch.tensor([[BOS_ID]], device=self.device)

        while len(out) < max_tokens:
          out = self.model.generate(context, out, max_tokens=10)[0].tolist()
          generated = out[1:]  # drop the leading <bos>
          if EOS_ID in generated:
              generated = generated[: generated.index(EOS_ID)]
              break
        return decode(generated)

# @title Shard-swap callback (relative OR absolute loss gap)

class ShardSwapCallback(pl.Callback):
    """Swap the training shard when the model starts overfitting it.

    Two complementary triggers, so we can swap even when losses are low:
      * absolute:  val_loss - train_loss > abs_gap        (early divergence)
      * relative:  train_loss / val_loss < ratio          (val >> train, low loss)

    On a swap we also notify the (optional) early-stopping callback so it grants
    the fresh shard a grace period before its val-loss regressions can count
    toward stopping. The swap callback must be listed BEFORE the early-stopping
    callback so the grace period is set before that round's stopping check runs.
    """

    def __init__(self, datamodule, abs_gap=0.30, ratio=0.80, early_stopping=None):
        super().__init__()
        self.datamodule = datamodule
        self.abs_gap = abs_gap
        self.ratio = ratio
        self.early_stopping = early_stopping  # GraceEarlyStopping or None

    def on_validation_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        metrics = trainer.callback_metrics
        train_loss = metrics.get("train_loss_epoch", metrics.get("train_loss"))
        val_loss = metrics.get("val_loss")
        if train_loss is None or val_loss is None:
            return

        t = float(train_loss)
        v = float(val_loss)
        gap = v - t
        ratio = t / v if v > 0 else 1.0
        absolute_trigger = gap > self.abs_gap
        relative_trigger = ratio < self.ratio

        print(f"[swap-check] step {trainer.global_step}: "
              f"train={t:.3f} val={v:.3f} gap={gap:.3f} t/v={ratio:.3f}")

        if absolute_trigger or relative_trigger:
            why = "abs gap" if absolute_trigger else "relative t/v"
            print(f"[swap-check] {why} triggered -> swapping training shard")
            self.datamodule.load_new_train_shard()
            # force Lightning (2.x) to rebuild the train dataloader from the new
            # shard; reload_dataloaders_every_n_epochs=1 keeps boundaries frequent
            trainer.fit_loop._combined_loader = None
            # give the new shard a grace period before early stopping can react
            if self.early_stopping is not None:
                self.early_stopping.start_grace()

# @title Early stopping with a post-swap grace period

class GraceEarlyStopping(pl.callbacks.EarlyStopping):
    """EarlyStopping that ignores val-loss regressions for `grace_period`
    validation checks after each shard swap.

    A swap changes the training distribution, so val_loss predictably bumps for
    a few checks afterwards. Without this, early stopping could fire on that
    transient. During the grace window we skip the stopping check entirely and
    keep the patience counter (`wait_count`) reset, so the fresh shard always
    gets a fair number of validations to settle before it can be stopped on.
    """

    def __init__(self, *args, grace_period=5, **kwargs):
        super().__init__(*args, **kwargs)
        self.grace_period = grace_period
        self._grace_remaining = 0

    def start_grace(self):
        # called by the swap callback when a new shard is loaded
        self._grace_remaining = self.grace_period

    def _run_early_stopping_check(self, trainer):
        if self._grace_remaining > 0:
            self._grace_remaining -= 1
            self.wait_count = 0          # don't accumulate patience during grace
            # still track the best score so post-grace comparisons are sensible
            logs = trainer.callback_metrics
            current = logs.get(self.monitor)
            if current is not None and self.monitor_op(current.squeeze(), self.best_score.to(current.device)):
                self.best_score = current.squeeze()
            print(f"[early-stop] grace period active "
                  f"({self._grace_remaining} checks left) -> not stopping")
            return
        super()._run_early_stopping_check(trainer)

# @title Train

# hyper parameters
NUM_LAYERS = 6
DIM = 384
NUM_HEADS = 6
MAX_CONTEXT = 192
P_DROP = 0.2
BATCH_SIZE = 128
LEARNING_RATE = 3e-4
VAL_CHECK_INTERVAL = 20
MAX_STEPS = 1_000

pl.seed_everything(42, workers=True)

datamodule = TinyStoriesDataModule(
    rows_per_shard=ROWS_PER_SHARD, validation_rows=VALIDATION_ROWS,
    batch_size=BATCH_SIZE, max_context=MAX_CONTEXT, seed=42)

model = SLMLightning(
    num_layers=NUM_LAYERS, dim=DIM, num_heads=NUM_HEADS, max_context=MAX_CONTEXT,
    vocab_size=VOCAB_SIZE, p_drop=P_DROP, learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS)

swap_cb = ShardSwapCallback(datamodule, abs_gap=0.30, ratio=0.80)
early_stop_cb = GraceEarlyStopping(
    monitor="val_loss", mode="min", patience=10, min_delta=1e-3,
    grace_period=5, verbose=True)
swap_cb.early_stopping = early_stop_cb        # let swaps trigger the grace period

trainer = pl.Trainer(
    max_steps=MAX_STEPS,
    val_check_interval=VAL_CHECK_INTERVAL,
    check_val_every_n_epoch=None,            # validate by step count
    # swap_cb MUST come before early_stop_cb so a swap sets the grace period
    # before that same validation round's stopping check runs.
    callbacks=[swap_cb, early_stop_cb],
    accelerator="auto",
    devices="auto",
    log_every_n_steps=25,
    gradient_clip_val=1.0,
    reload_dataloaders_every_n_epochs=1,     # honour mid-run shard swaps
)

trainer.fit(model, datamodule=datamodule)


# @title Sample generations

for ctx in [
    "Once upon a time there was a little girl.",
    "Tom had a red ball. He kicked it high.",
    "The dog was very hungry.",
]:
    print(f"context : {ctx}")
    print(f"next    : {model.generate_next_sentence(ctx)}\n")

# @title Sample generations

for ctx in [
    "Once upon a time there was a little girl.",
    "Tom had a red ball. He kicked it high.",
    "The dog was very hungry.",
]:
    print(f"context : {ctx}")
    print(f"next    : {model.generate_next_sentence(ctx)}\n")